# Optativa III: Ciencia de Datos
## Actividad práctica — Probabilidad aplicada, teorema de Bayes e inferencia

**Universidad de Especialidades UNE · Plantel Centro**
**Ingeniería en Computación · 9.º semestre · Semana 2 · Sesión 2**

---

### Propósito

En esta actividad aplicarás la **probabilidad condicional**, el **teorema de Bayes**, el concepto de **variables aleatorias** y los **intervalos de confianza** en dos partes:

- **Parte A —** Construirás un **filtro de spam conceptual** con el teorema de Bayes, paso a paso, y experimentarás modificando sus parámetros.
- **Parte B —** Resolverás un **set de 6 problemas de probabilidad e inferencia** sobre un **dataset real** de un restaurante.

A lo largo del cuaderno responderás **20 preguntas** (marcadas con ✍️). No basta con ejecutar: en cada bloque deberás **modificar valores, volver a correr y explicar** lo que observas.

### Cómo trabajar en Google Colab

1. Abre este archivo en [Google Colab](https://colab.research.google.com) (Archivo → Subir cuaderno).
2. Ejecuta cada celda con **Shift + Enter**, en orden.
3. Responde en las celdas de texto ✍️ (doble clic para editar).
4. Al terminar: **Archivo → Guardar una copia en Drive** y entrega el enlace en Classroom.


---
## Paso 0 — Preparación del entorno


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
print("Entorno listo. Librerias importadas correctamente.")

> ✍️ **Pregunta 1** — Explica con tus palabras la diferencia entre **probabilidad simple** P(A) y **probabilidad condicional** P(A|B). Da un ejemplo cotidiano de cada una.


✍️ **Respuesta 1**

-   **Probabilidad Simple P(A):** Es la probabilidad de que un evento ocurra por sí mismo, sin ninguna otra condición. Es la relación entre el número de resultados favorables a ese evento y el número total de resultados posibles.

    *   **Ejemplo cotidiano:** La probabilidad de que llueva un día cualquiera. Si históricamente llueve 100 días al año, la probabilidad es 100/365 (aproximadamente 0.27 o 27%).

-   **Probabilidad Condicional P(A|B):** Es la probabilidad de que el evento A ocurra, sabiendo que otro evento B ya ha ocurrido. Se lee como "la probabilidad de A dado B". El evento B restringe el espacio muestral a considerar.

    *   **Ejemplo cotidiano:** La probabilidad de que llueva **dado que** el cielo está nublado. Si en todos los días nublados llueve el 70% de las veces, entonces P(llueva | nublado) = 0.70. Aquí, el hecho de que el cielo esté nublado cambia nuestra expectativa sobre la lluvia.

---
# PARTE A — Filtro de spam con el teorema de Bayes

## Paso 1 — El problema

Queremos clasificar un correo como **spam** o **legítimo (ham)** según las palabras que contiene. El teorema de Bayes nos permite calcular la probabilidad de que un correo sea spam **dado que** contiene cierta palabra:

$$P(spam \mid palabra) = \frac{P(palabra \mid spam) \cdot P(spam)}{P(palabra)}$$

donde:
- **P(spam)** es la probabilidad previa (qué proporción de correos es spam en general).
- **P(palabra | spam)** es la verosimilitud (qué tan frecuente es la palabra en el spam).
- **P(palabra)** es la probabilidad total de ver la palabra.


In [ ]:
# Datos base del problema (probabilidades conocidas por experiencia)
P_spam = 0.40          # 40% de los correos que llegan son spam
P_ham  = 1 - P_spam    # el resto son legitimos

# Verosimilitudes: que tan seguido aparece la palabra "gratis" en cada tipo
P_gratis_dado_spam = 0.80   # el 80% de los spam contienen "gratis"
P_gratis_dado_ham  = 0.10   # solo el 10% de los correos legitimos la usan

print(f"P(spam)              = {P_spam}")
print(f"P(ham)               = {P_ham}")
print(f"P(gratis | spam)     = {P_gratis_dado_spam}")
print(f"P(gratis | ham)      = {P_gratis_dado_ham}")

> ✍️ **Pregunta 2** — En este modelo, ¿cuál es la **probabilidad previa** (prior) de que un correo sea spam? ¿De dónde crees que sale ese número en un sistema real?


## Paso 2 — Aplicando el teorema de Bayes

Calculamos la probabilidad total de la palabra con el **teorema de la probabilidad total** y luego aplicamos Bayes.


In [ ]:
# Teorema de la probabilidad total: P(gratis)
P_gratis = P_gratis_dado_spam * P_spam + P_gratis_dado_ham * P_ham

# Teorema de Bayes: P(spam | gratis)
P_spam_dado_gratis = (P_gratis_dado_spam * P_spam) / P_gratis

print(f"P(gratis)               = {P_gratis:.4f}")
print(f"P(spam | 'gratis')      = {P_spam_dado_gratis:.4f}  ->  {P_spam_dado_gratis*100:.1f}%")
print()
if P_spam_dado_gratis > 0.5:
    print("Decision: clasificar como SPAM")
else:
    print("Decision: clasificar como LEGITIMO")

> ✍️ **Pregunta 3** — Ejecuta la celda. ¿Cuál es la probabilidad de que un correo con la palabra "gratis" sea spam? ¿El sistema lo clasificaría como spam?

> ✍️ **Pregunta 4** — Compara P(gratis | spam) = 0.80 con P(spam | gratis) que obtuviste. **No son iguales.** Explica por qué invertir la condición cambia el resultado (esto se conoce como la "falacia de la condicional transpuesta").


## Paso 3 — Función reutilizable del clasificador

Convertimos el cálculo en una función para poder probar distintas palabras y parámetros con facilidad.


In [ ]:
def clasificar_bayes(P_spam, P_palabra_dado_spam, P_palabra_dado_ham):
    """Devuelve P(spam | palabra) usando el teorema de Bayes."""
    P_ham = 1 - P_spam
    P_palabra = P_palabra_dado_spam * P_spam + P_palabra_dado_ham * P_ham
    return (P_palabra_dado_spam * P_spam) / P_palabra

# Probamos con varias palabras (cada una con su verosimilitud)
palabras = {
    "gratis":   (0.80, 0.10),
    "reunion":  (0.05, 0.40),
    "oferta":   (0.70, 0.15),
    "proyecto": (0.03, 0.50),
    "premio":   (0.85, 0.05),
}

print(f"{'Palabra':<12}{'P(spam|palabra)':>18}")
print("-"*30)
for palabra, (p_spam, p_ham) in palabras.items():
    resultado = clasificar_bayes(0.40, p_spam, p_ham)
    etiqueta = "SPAM" if resultado > 0.5 else "legitimo"
    print(f"{palabra:<12}{resultado*100:>15.1f}%   {etiqueta}")

> ✍️ **Pregunta 5** — Según la tabla, ¿qué palabras hacen que un correo sea clasificado como spam y cuáles como legítimo? ¿Tiene sentido con tu intuición?

> ✍️ **Pregunta 6** — La palabra "reunion" da una probabilidad de spam **baja**. Explica qué característica de sus verosimilitudes (P(palabra|spam) vs P(palabra|ham)) produce ese resultado.


### 🔧 Reto de modificación 1

Agrega **dos palabras nuevas** al diccionario `palabras` (por ejemplo "descuento" y "tarea") con las verosimilitudes que consideres realistas, y vuelve a ejecutar.


In [ ]:
# RETO: copia el diccionario, agrega tus dos palabras y ejecuta
palabras_ampliado = {
    "gratis":   (0.80, 0.10),
    "reunion":  (0.05, 0.40),
    # agrega aqui tus dos palabras nuevas:
    # "descuento": (?, ?),
    # "tarea":     (?, ?),
}

for palabra, (p_spam, p_ham) in palabras_ampliado.items():
    r = clasificar_bayes(0.40, p_spam, p_ham)
    print(f"{palabra:<12}{r*100:>6.1f}%   {'SPAM' if r>0.5 else 'legitimo'}")

> ✍️ **Pregunta 7** — ¿Qué verosimilitudes elegiste para tus dos palabras y por qué? ¿El clasificador coincidió con lo que esperabas?


## Paso 4 — El efecto de la probabilidad previa (prior)

La probabilidad previa P(spam) tiene un efecto enorme en el resultado. Veamos qué pasa si cambia la proporción de spam que recibe el servidor.


In [ ]:
# Mantenemos la palabra "gratis" (0.80 en spam, 0.10 en ham)
# y variamos el prior P(spam)
priors = [0.05, 0.20, 0.40, 0.60, 0.90]

resultados = []
for prior in priors:
    r = clasificar_bayes(prior, 0.80, 0.10)
    resultados.append(r)
    print(f"P(spam)={prior:.2f}  ->  P(spam | 'gratis') = {r*100:.1f}%")

plt.figure(figsize=(9,5))
plt.plot(priors, [r*100 for r in resultados], "o-", color="#1e2a6b", linewidth=2, markersize=8)
plt.axhline(50, color="#e2231a", linestyle="--", label="Umbral de decision (50%)")
plt.title("Efecto del prior P(spam) sobre la clasificacion de 'gratis'")
plt.xlabel("Probabilidad previa P(spam)")
plt.ylabel("P(spam | 'gratis')  (%)")
plt.legend(); plt.grid(alpha=0.2); plt.show()

> ✍️ **Pregunta 8** — Cuando P(spam) es muy baja (0.05), ¿la palabra "gratis" basta para clasificar el correo como spam? ¿Qué nos enseña esto sobre la importancia del contexto (el prior)?

> ✍️ **Pregunta 9** — Imagina un servidor corporativo donde solo el 5% de los correos es spam, frente a una cuenta personal donde el 60% lo es. ¿Por qué el **mismo correo** podría clasificarse distinto en cada uno? Relaciónalo con la gráfica.


## Paso 5 — Combinando varias palabras (Naive Bayes)

Un filtro real no mira una sola palabra. El clasificador **Naive Bayes** combina la evidencia de varias palabras asumiendo (de forma "ingenua") que son independientes. Aquí lo hacemos con dos palabras.


In [ ]:
def naive_bayes_dos_palabras(P_spam, verosim1, verosim2):
    """Clasifica usando dos palabras simultaneamente.
       verosim = (P(palabra|spam), P(palabra|ham))"""
    P_ham = 1 - P_spam
    # Verosimilitud conjunta asumiendo independencia
    L_spam = verosim1[0] * verosim2[0] * P_spam
    L_ham  = verosim1[1] * verosim2[1] * P_ham
    return L_spam / (L_spam + L_ham)

# Un correo que contiene "gratis" Y "premio"
p = naive_bayes_dos_palabras(0.40, (0.80, 0.10), (0.85, 0.05))
print(f"Correo con 'gratis' Y 'premio':  P(spam) = {p*100:.2f}%")

# Un correo con "gratis" pero tambien "proyecto"
p2 = naive_bayes_dos_palabras(0.40, (0.80, 0.10), (0.03, 0.50))
print(f"Correo con 'gratis' Y 'proyecto': P(spam) = {p2*100:.2f}%")

> ✍️ **Pregunta 10** — Un correo con "gratis" solo daba ~84% de spam. Al añadir "premio", ¿la probabilidad sube o baja? ¿Y al añadir "proyecto"? Explica cómo la evidencia se **acumula** en Naive Bayes.

> ✍️ **Pregunta 11** — ¿Por qué se le llama "ingenuo" (naive) a este clasificador? ¿Qué suposición hace que rara vez es 100% cierta en el lenguaje real? (Pista: piensa en palabras que casi siempre aparecen juntas.)


---
# PARTE B — Problemas de probabilidad e inferencia sobre un dataset real

## Paso 6 — Cargamos el dataset

Usaremos el dataset **`tips`**, con **244 registros reales** de propinas de un restaurante. Incluye la cuenta total, la propina, el sexo del cliente, si es fumador, el día y el momento (comida/cena) y el tamaño del grupo. Se carga directo desde `seaborn`, sin descargas.


In [ ]:
import seaborn as sns
df = sns.load_dataset("tips")

print("Dimensiones:", df.shape)
print()
print(df.head())
print()
print("Resumen de variables categoricas:")
for col in ["sex", "smoker", "day", "time"]:
    print(f"  {col}: {df[col].unique().tolist()}")

> ✍️ **Pregunta 12** — Describe el dataset con tus palabras: ¿cuántos registros tiene, qué representa cada fila y qué variables son **categóricas** y cuáles **numéricas**?


## Problema 1 — Probabilidad simple

Calculamos probabilidades básicas como frecuencias relativas.


In [ ]:
total = len(df)

# P(cliente sea fumador)
p_fumador = (df["smoker"] == "Yes").sum() / total
# P(la visita sea en la cena)
p_cena = (df["time"] == "Dinner").sum() / total

print(f"P(fumador)      = {p_fumador:.4f}  ({p_fumador*100:.1f}%)")
print(f"P(cena)         = {p_cena:.4f}  ({p_cena*100:.1f}%)")

> ✍️ **Pregunta 13** — Calcula tú, modificando el código, la probabilidad de que **el cliente sea hombre** P(sex == "Male") y la de que la visita sea **en fin de semana** (day en ["Sat", "Sun"]). Escribe el código y reporta ambos valores.


In [ ]:
# RETO: escribe aqui tu codigo para las dos probabilidades pedidas
# p_hombre = ...
# p_finde  = ...


## Problema 2 — Probabilidad condicional

Ahora condicionamos: la probabilidad de un evento **dado que** otro ocurrió.


In [ ]:
# P(fumador | es cena)  =  P(fumador Y cena) / P(cena)
cena = df[df["time"] == "Dinner"]
p_fumador_dado_cena = (cena["smoker"] == "Yes").sum() / len(cena)

print(f"P(fumador | cena) = {p_fumador_dado_cena:.4f}  ({p_fumador_dado_cena*100:.1f}%)")

# Comparamos con la probabilidad no condicionada
print(f"P(fumador)        = {(df['smoker']=='Yes').mean():.4f}  (para comparar)")

> ✍️ **Pregunta 14** — ¿La probabilidad de ser fumador **cambia** al condicionar por "cena"? Si P(fumador|cena) ≈ P(fumador), ¿qué sugiere sobre la relación entre ambas variables?

> ✍️ **Pregunta 15** — Modifica el código para calcular **P(cena | fumador)**. ¿Es igual a P(fumador | cena)? Explica por qué el orden de la condición importa.


In [ ]:
# RETO: calcula P(cena | fumador)
# fumadores = df[df["smoker"] == "Yes"]
# p_cena_dado_fumador = ...


## Problema 3 — Teorema de Bayes sobre el dataset

Aplicamos Bayes con datos reales para invertir una condicional.


In [ ]:
# Queremos P(cena | grupo grande), definiendo "grupo grande" = size >= 4
# Bayes:  P(cena | grande) = P(grande | cena) * P(cena) / P(grande)

df["grande"] = df["size"] >= 4

P_cena     = (df["time"] == "Dinner").mean()
P_grande   = df["grande"].mean()
P_grande_dado_cena = df[df["time"]=="Dinner"]["grande"].mean()

P_cena_dado_grande = (P_grande_dado_cena * P_cena) / P_grande

print(f"P(cena)                = {P_cena:.4f}")
print(f"P(grupo grande)        = {P_grande:.4f}")
print(f"P(grande | cena)       = {P_grande_dado_cena:.4f}")
print(f"P(cena | grupo grande) = {P_cena_dado_grande:.4f}  <- via Bayes")

# Verificacion directa
verif = (df[df["grande"]]["time"] == "Dinner").mean()
print(f"Verificacion directa   = {verif:.4f}")

> ✍️ **Pregunta 16** — El resultado de Bayes y la verificación directa deben coincidir. ¿Coinciden? ¿Qué te dice esto sobre la validez del teorema de Bayes como una forma de "reorganizar" la información?


## Problema 4 — Variables aleatorias: valor esperado y varianza

La propina (`tip`) es una **variable aleatoria continua**. Calculamos su valor esperado (media), varianza y desviación estándar.


In [ ]:
tip = df["tip"]

esperanza = tip.mean()      # E[X], valor esperado
varianza  = tip.var(ddof=1) # Var(X)
desv      = tip.std(ddof=1)

print(f"E[propina]   (valor esperado) = ${esperanza:.2f}")
print(f"Var(propina)                  = {varianza:.2f}")
print(f"Desv. estandar                = ${desv:.2f}")

plt.figure(figsize=(9,5))
plt.hist(tip, bins=20, color="#1e2a6b", alpha=0.7, edgecolor="white")
plt.axvline(esperanza, color="#e2231a", linestyle="--", linewidth=2,
            label=f"E[propina] = ${esperanza:.2f}")
plt.title("Distribucion de la propina (variable aleatoria)")
plt.xlabel("Propina ($)"); plt.ylabel("Frecuencia")
plt.legend(); plt.grid(alpha=0.2); plt.show()

> ✍️ **Pregunta 17** — Interpreta el **valor esperado** de la propina en el contexto del restaurante. Si atienden 500 mesas en un mes, ¿cuánto esperarían recaudar en propinas aproximadamente? ¿Qué operación usaste?


## Problema 5 — Intervalo de confianza para la media

Estimamos un **intervalo de confianza del 95%** para la propina promedio. Este rango expresa la incertidumbre de estimar la media poblacional a partir de la muestra.


In [ ]:
n         = len(tip)
media     = tip.mean()
error_est = tip.std(ddof=1) / np.sqrt(n)   # error estandar de la media

# Intervalo de confianza del 95% usando la distribucion t de Student
confianza = 0.95
ic = stats.t.interval(confianza, df=n-1, loc=media, scale=error_est)

print(f"Tamano de muestra (n)       = {n}")
print(f"Media muestral              = ${media:.3f}")
print(f"Error estandar              = ${error_est:.3f}")
print(f"IC del 95% para la media    = (${ic[0]:.3f}, ${ic[1]:.3f})")
print()
print(f"Interpretacion: con 95% de confianza, la propina promedio real")
print(f"esta entre ${ic[0]:.2f} y ${ic[1]:.2f}.")

> ✍️ **Pregunta 18** — Explica con tus palabras qué significa el intervalo de confianza del 95% que obtuviste. ¿Es correcto decir "hay 95% de probabilidad de que la media esté en este rango"? (Investiga la interpretación correcta.)

> ✍️ **Pregunta 19** — Modifica el nivel de confianza a **99%** (cambia `confianza = 0.99`) y vuelve a ejecutar. ¿El intervalo se hace más **ancho** o más **angosto**? ¿Por qué mayor confianza implica ese cambio?


In [ ]:
# RETO: calcula el intervalo al 99% y comparalo con el del 95%
# ic_99 = stats.t.interval(0.99, df=n-1, loc=media, scale=error_est)
# print(ic_99)


## Problema 6 — Inferencia: comparación entre grupos

¿Los fumadores dejan propinas distintas que los no fumadores? Comparamos las medias y sus intervalos de confianza.


In [ ]:
def ic_media(serie, confianza=0.95):
    n = len(serie)
    m = serie.mean()
    ee = serie.std(ddof=1) / np.sqrt(n)
    lo, hi = stats.t.interval(confianza, df=n-1, loc=m, scale=ee)
    return m, lo, hi

tip_fuma    = df[df["smoker"]=="Yes"]["tip"]
tip_no_fuma = df[df["smoker"]=="No"]["tip"]

m1, lo1, hi1 = ic_media(tip_fuma)
m2, lo2, hi2 = ic_media(tip_no_fuma)

print(f"Fumadores    : media=${m1:.2f}  IC95%=(${lo1:.2f}, ${hi1:.2f})  n={len(tip_fuma)}")
print(f"No fumadores : media=${m2:.2f}  IC95%=(${lo2:.2f}, ${hi2:.2f})  n={len(tip_no_fuma)}")

plt.figure(figsize=(8,5))
grupos = ["Fumadores", "No fumadores"]
medias = [m1, m2]
errores = [[m1-lo1, m2-lo2],[hi1-m1, hi2-m2]]
plt.bar(grupos, medias, yerr=errores, capsize=10,
        color=["#e2231a","#1e2a6b"], alpha=0.8, edgecolor="white")
plt.title("Propina promedio por grupo (con IC del 95%)")
plt.ylabel("Propina promedio ($)")
plt.grid(alpha=0.2, axis="y"); plt.show()

> ✍️ **Pregunta 20** — Observa si los intervalos de confianza de ambos grupos **se traslapan**. Si se traslapan mucho, no podemos afirmar que haya una diferencia real. Según tu gráfica, ¿dirías que fumadores y no fumadores dejan propinas distintas? Justifica tu respuesta con los intervalos.


---
## Cierre — Reflexión y entrega

> **Reflexión final** (escribe un párrafo): De todo lo trabajado (probabilidad condicional, teorema de Bayes, variables aleatorias e intervalos de confianza), ¿qué concepto te parece más útil para tu futuro como ingeniero en computación y en qué tipo de sistema lo aplicarías? Da un ejemplo concreto (por ejemplo: detección de fraude, sistemas de recomendación, control de calidad, diagnóstico automático).

### Entrega en Google Classroom

1. Verifica que **todas las celdas corran sin error** (Entorno de ejecución → Ejecutar todo).
2. Confirma que respondiste las **20 preguntas ✍️**, los retos de código y la reflexión.
3. **Archivo → Guardar una copia en Drive** y comparte el enlace en Classroom.

### Rúbrica

| Criterio | Puntos |
|---|---|
| Todas las celdas se ejecutan correctamente | 15 |
| Retos de código resueltos (Parte A y B) | 25 |
| Respuestas a las 20 preguntas ✍️ | 40 |
| Reflexión final argumentada | 10 |
| Orden y documentación del notebook | 10 |
| **Total** | **100** |

---
*Universidad de Especialidades UNE · Plantel Centro · Optativa III: Ciencia de Datos*
